In [0]:
#programa para leer los documentos en la carpeta e-commerce

import os

ruta = '/Volumes/workspace/default/ecommerce_raw'

archivos = os.listdir(ruta)

print("="*31)
print("LECTURA DE ARCHIVOS ALMACENADOS\nEN CARPETA ecommerce_raw")
print("="*31)


for f in archivos:
    size = os.path.getsize(f'{ruta}/{f}')
    print(f"📄 {f}  →  {size/1e9:.2f} GB")

print()

# Capa Bronze
Almacenaremos los datos crudos como vienen desde Kaggle

In [0]:
# 01_Ingestion_Bronze_V3.ipynb
from pyspark.sql.types import StructType, StructField, StringType, FloatType
from pyspark.sql.functions import current_timestamp, col

# 1. Crear el Volumen de Unity Catalog programáticamente
print("1. Verificando/Creando Volumen 'e_commerce'...")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.e_commerce")

# 2. Definición estricta de esquema
schema = StructType([
    StructField("event_time", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("category_id", StringType(), True),
    StructField("category_code", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("price", FloatType(), True),
    StructField("user_id", StringType(), True),
    StructField("user_session", StringType(), True)
])

# 3. Definición de rutas según tu nueva arquitectura
# Leemos de donde descargaste originalmente, escribimos en la nueva estructura
raw_data_path = "/Volumes/workspace/default/ecommerce_raw/*.csv"
bronze_table_path = "/Volumes/workspace/default/e_commerce/bronze/clickstream"

# 4. Lectura de los CSVs optimizada
print("2. Leyendo datos Raw...")
df_raw = spark.read.format("csv") \
    .option("header", "true") \
    .schema(schema) \
    .load(raw_data_path)

# 5. Añadir metadatos de linaje (Unity Catalog)
print("3. Añadiendo metadatos de linaje...")
df_bronze = df_raw \
    .withColumn("_ingest_timestamp", current_timestamp()) \
    .withColumn("_source_file", col("_metadata.file_path"))

# 6. Escritura en formato Delta (Capa Bronze)
# Databricks creará automáticamente la subcarpeta 'bronze/clickstream' dentro del Volumen 'e_commerce'
print("4. Escribiendo en formato Delta (Capa Bronze)... Esto tomará unos minutos.")
(df_bronze.write
    .format("delta")
    .mode("append") 
    .save(bronze_table_path))

print("✅ Pipeline ejecutado con éxito. Datos listos en capa Bronze.")

# Silver
En esta capa, tomaremos los datos crudos de la capa Bronze y aplicaremos limpieza, tipado de datos (casting) y extracción de características básicas (parsing).
Objetivos de esta transformación:Tipado Temporal: 
1. Convertir event_time de String a Timestamp para poder calcular recencias matemáticas.
2. Parsing de Taxonomía: La columna category_code viene concatenada (ej. electronics.smartphone.apple). Debemos separarla para que los modelos puedan encontrar patrones por macro-categoría.  
3. Streaming Sink: Escribiremos el resultado en la carpeta silver dentro del volumen, usando un checkpoint para que Spark sepa exactamente qué datos ya procesó en caso de que el clúster se apague.